# 🛰️ Image Classification with Convolutional Neural Networks

> **Activity:** Classify satellite images of Mexican biomes using two approaches — a Multilayer Perceptron with color histogram features and a Convolutional Neural Network — and compare their performance with cross-validation.

---

## Table of Contents
1. [Setup & Imports](#setup)
2. [Load Dataset](#load)
3. [Exercise 1 — MLP with Color Histograms](#exercise-1)
4. [Exercise 2 — Convolutional Neural Network](#exercise-2)
5. [Exercise 3 — Performance Comparison](#exercise-3)
6. [Overall Conclusions](#conclusions)

---
**Dataset:** Satellite images of six Mexican biomes  
**Classes:** Agua, Bosque, Ciudad, Cultivo, Desierto, Montaña  
**Expected structure:** `Biomas/<biome_name>/*.jpg`

> Adjust `DATASET_DIR` in the Setup cell to point to your `Biomas` folder.

---
<a id='setup'></a>
## ⚙️ 1. Setup & Imports

In [ ]:
# Install required packages (run once if needed)
# !pip install tensorflow scikit-learn matplotlib numpy opencv-python seaborn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
import os, time
from pathlib import Path
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Image processing
import cv2

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

# Classical ML
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_validate
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, precision_score, recall_score
)
import seaborn as sns

# Plot styling
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'grid.linestyle':   '--',
    'font.family':      'sans-serif',
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

COLORS = ['#2E5C8A', '#DD8452', '#55A868', '#C44E52', '#8172B2']

# ── Global configuration ───────────────────────────────
DATASET_DIR  = '../Biomas'      # ← Adjust this path if needed
IMG_SIZE     = (48, 48)         # Resize all images to this
N_BINS       = 32               # Bins per channel for histograms
N_SPLITS     = 5                # K-fold cross-validation folds
RANDOM_STATE = 42

print('✅ All imports successful!')
print(f'   TensorFlow version : {tf.__version__}')
print(f'   OpenCV version     : {cv2.__version__}')
print(f'   Dataset directory  : {DATASET_DIR}')

---
<a id='load'></a>
## 📁 2. Load Dataset

Single pass through the dataset extracting **both** color histograms (for the MLP) and raw images (for the CNN). This saves I/O time later.

> **Note on Windows:** `cv2.imread()` cannot handle paths with non-ASCII characters (such as the `ñ` in `Montaña`). The `safe_imread()` helper below uses `numpy.fromfile()` to read the file bytes first and then `cv2.imdecode()` to decode them, which works regardless of path encoding.

In [ ]:
def safe_imread(path):
    """Unicode-safe image reader (works with paths containing ñ, accents, etc.)."""
    try:
        data = np.fromfile(str(path), dtype=np.uint8)
        return cv2.imdecode(data, cv2.IMREAD_COLOR) if data.size else None
    except Exception:
        return None


def list_images(folder):
    return (list(folder.glob('*.jpg')) + list(folder.glob('*.jpeg')) +
            list(folder.glob('*.png')) + list(folder.glob('*.JPG')))


def compute_histogram(img_rgb, n_bins=N_BINS):
    return np.concatenate([
        np.histogram(img_rgb[:, :, c], bins=n_bins,
                     range=(0, 256), density=True)[0]
        for c in range(3)
    ])


def load_biomes_dataset(root_dir):
    """Loads images once, returning histograms, normalized images, and labels."""
    root = Path(root_dir)
    biomes = sorted([p.name for p in root.iterdir() if p.is_dir()])
    X_hist, X_img, y = [], [], []

    print(f'Loading from: {root.resolve()}')
    print(f'Classes detected: {biomes}\n')

    for biome in biomes:
        folder = root / biome
        imgs = list_images(folder)
        loaded = 0
        for img_path in imgs:
            img = safe_imread(img_path)
            if img is None:
                continue
            img = cv2.cvtColor(cv2.resize(img, IMG_SIZE), cv2.COLOR_BGR2RGB)
            X_hist.append(compute_histogram(img))
            X_img.append(img.astype(np.float32) / 255.0)
            y.append(biome)
            loaded += 1
        print(f'  {biome:12s}: {loaded}/{len(imgs)} images')

    return np.array(X_hist), np.array(X_img), np.array(y), biomes


print('✅ Loading helpers defined.')

In [ ]:
if not Path(DATASET_DIR).exists():
    raise FileNotFoundError(
        f"Could not find '{DATASET_DIR}'. Adjust DATASET_DIR in the Setup cell.")

t0 = time.time()
X_hist, X_img, y_str, biomes = load_biomes_dataset(DATASET_DIR)
print(f'\nLoading time: {time.time()-t0:.1f}s')

le = LabelEncoder()
y = le.fit_transform(y_str)
class_names = le.classes_

print('=' * 55)
print('DATASET OVERVIEW')
print('=' * 55)
print(f'Total observations  : {X_img.shape[0]}')
print(f'Image shape         : {X_img.shape[1:]}')
print(f'Histogram features  : {X_hist.shape[1]}  ({N_BINS} bins × 3 channels)')
print(f'Number of classes   : {len(class_names)}')
print('\nClass distribution:')
for cls, count in zip(*np.unique(y_str, return_counts=True)):
    print(f'  {cls:12s}: {count} images')

In [ ]:
# Preview one example per biome
fig, axes = plt.subplots(1, len(class_names), figsize=(15, 3))
for ax, cls in zip(axes, class_names):
    idx = np.where(y_str == cls)[0][0]
    ax.imshow(X_img[idx])
    ax.set_title(cls, fontsize=10, fontweight='bold')
    ax.axis('off')
plt.suptitle('Sample image per biome', fontsize=12, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig('Dataset_preview.png', dpi=150, bbox_inches='tight')
plt.show()

---
<a id='exercise-1'></a>
## 🎨 Exercise 1 — MLP with Color Histograms

**Goal:** Build a multilayer perceptron classifier using **color histograms** as features and evaluate it with cross-validation.

**Feature extraction:**
- Resize image to 48×48
- Compute a 32-bin normalized histogram per RGB channel
- Concatenate into a 96-dimensional feature vector

**Architecture:** `Input(96) → Dense(256, ReLU) → Dense(128, ReLU) → Dense(6, Softmax)`  
**Optimizer:** Adam | **Loss:** Cross-Entropy | **Early Stopping:** patience=15  
**Evaluation:** 5-fold stratified cross-validation

### 1.1 Build MLP Pipeline (Scaler + Classifier)

In [ ]:
def build_mlp_pipeline():
    return Pipeline([
        ('scaler', StandardScaler()),
        ('mlp', MLPClassifier(
            hidden_layer_sizes=(256, 128),
            activation='relu',
            solver='adam',
            max_iter=500,
            learning_rate_init=1e-3,
            random_state=RANDOM_STATE,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=15,
            verbose=False
        ))
    ])

print('✅ MLP pipeline defined.')
print('   Architecture: Dense(256,ReLU) → Dense(128,ReLU) → Dense(6,Softmax)')

### 1.2 Five-Fold Stratified Cross-Validation

In [ ]:
kf1 = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

print(f'Training MLP — {N_SPLITS}-fold stratified CV')
print(f'Features: color histograms ({N_BINS} bins × 3 channels = {N_BINS*3}-D)\n')

t_mlp = time.time()
cv_results_mlp = cross_validate(
    build_mlp_pipeline(), X_hist, y,
    cv=kf1,
    scoring=['accuracy', 'f1_macro', 'precision_macro', 'recall_macro'],
    return_train_score=True,
    n_jobs=-1
)
mlp_cv_time = time.time() - t_mlp

print(f'{"Fold":>6}  {"Train Acc":>11}  {"Val Acc":>10}  {"F1 Macro":>10}')
print('-' * 45)
for i in range(N_SPLITS):
    print(f'{i+1:>6}  '
          f'{cv_results_mlp["train_accuracy"][i]*100:>10.2f}%  '
          f'{cv_results_mlp["test_accuracy"][i]*100:>9.2f}%  '
          f'{cv_results_mlp["test_f1_macro"][i]*100:>9.2f}%')

mlp_acc_mean = cv_results_mlp['test_accuracy'].mean()
mlp_acc_std  = cv_results_mlp['test_accuracy'].std()
mlp_f1_mean  = cv_results_mlp['test_f1_macro'].mean()
mlp_f1_std   = cv_results_mlp['test_f1_macro'].std()

print(f'\nMean CV Accuracy : {mlp_acc_mean:.4f} ± {mlp_acc_std:.4f}')
print(f'Mean CV F1 Macro : {mlp_f1_mean:.4f} ± {mlp_f1_std:.4f}')
print(f'CV Time          : {mlp_cv_time:.1f}s')

### 1.3 Final Model — 80/20 Split for Detailed Report

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X_hist, y, test_size=0.2,
    stratify=y, random_state=RANDOM_STATE
)

mlp_final = build_mlp_pipeline()
mlp_final.fit(X_tr, y_tr)
y_pred_mlp = mlp_final.predict(X_te)

# Save metrics for Exercise 3
mlp_metrics = {
    'name'          : 'MLP + Histograms',
    'cv_accs'       : cv_results_mlp['test_accuracy'],
    'cv_acc_mean'   : mlp_acc_mean,
    'cv_acc_std'    : mlp_acc_std,
    'cv_f1_mean'    : mlp_f1_mean,
    'cv_f1_std'     : mlp_f1_std,
    'cv_time'       : mlp_cv_time,
    'y_te'          : y_te,
    'y_pred'        : y_pred_mlp,
    'test_acc'      : accuracy_score(y_te, y_pred_mlp),
    'test_f1'       : f1_score(y_te, y_pred_mlp, average='macro'),
    'test_precision': precision_score(y_te, y_pred_mlp, average='macro'),
    'test_recall'   : recall_score(y_te, y_pred_mlp, average='macro'),
}

print('Classification Report (test set, 20%):')
print(classification_report(y_te, y_pred_mlp, target_names=class_names))

### 1.4 Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy per fold
ax = axes[0]
folds = np.arange(1, N_SPLITS + 1)
ax.bar(folds - 0.18, cv_results_mlp['train_accuracy']*100, width=0.36,
       color=COLORS[0], alpha=0.85, edgecolor='black', label='Train')
ax.bar(folds + 0.18, cv_results_mlp['test_accuracy']*100, width=0.36,
       color=COLORS[2], alpha=0.85, edgecolor='black', label='Validation')
ax.axhline(mlp_acc_mean*100, color=COLORS[3], linestyle='--',
           label=f'Mean Val = {mlp_acc_mean*100:.1f}%')
ax.set_xticks(folds)
ax.set_xlabel('Fold'); ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 105)
ax.set_title('Accuracy per Fold (5-fold CV)', fontsize=12, fontweight='bold')
ax.legend(loc='lower right')

# Confusion matrix
ax = axes[1]
cm_mlp = confusion_matrix(y_te, y_pred_mlp)
sns.heatmap(cm_mlp, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — MLP', fontsize=12, fontweight='bold')
plt.setp(ax.get_xticklabels(), rotation=40, ha='right')

plt.suptitle('Exercise 1 — MLP with Color Histograms', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('Exercise1.png', dpi=150, bbox_inches='tight')
plt.show()

### 📝 Exercise 1 — Conclusions

The MLP classifier with color histograms achieves a stable accuracy across folds, with low variance — confirming that the global color signature alone provides a strong baseline for biome discrimination.

**Observations:**
- Histograms compress each image into a compact 96-dimensional vector, regardless of the image's resolution or spatial layout.
- Biomes with distinctive color palettes (Water → blue, Desert → warm tones, Forest → greens) are easiest to separate.
- Confusion is more likely between biomes that share dominant colors (e.g., Forest vs. Crop — both green-dominant; Mountain vs. City — both gray/brown).
- The main limitation is loss of **spatial information**: two images with the same color distribution but different structure look identical to this model.

This is exactly the gap that the CNN in Exercise 2 will try to fill.

---
<a id='exercise-2'></a>
## 🧠 Exercise 2 — Convolutional Neural Network

**Goal:** Build a CNN that classifies the same biome images directly from raw pixels, capturing spatial patterns the histogram-based MLP cannot see.

> ⚠️ Implemented in **Keras (TensorFlow)** as required.

**Input:** 48×48 RGB images (small to keep training feasible on CPU).

**Architecture:**
```
Input(48, 48, 3)
  → Conv2D(16, 3×3) → ReLU → MaxPool(2×2) → Dropout(0.25)
  → Conv2D(32, 3×3) → ReLU → MaxPool(2×2) → Dropout(0.25)
  → Flatten → Dense(64, ReLU) → Dropout(0.4)
  → Dense(6, Softmax)
```

**Optimizer:** Adam (lr=1e-3) | **Loss:** Sparse Categorical Crossentropy  
**Epochs:** 15 (with EarlyStopping patience=4) | **Batch size:** 64  
**Evaluation:** 5-fold stratified cross-validation

### 2.1 Build CNN Model

In [ ]:
EPOCHS_CNN = 15
BATCH_SIZE = 64
N_CLASSES  = len(class_names)

def build_cnn_model(input_shape, n_classes):
    model = keras.Sequential([
        keras.Input(shape=input_shape),
        layers.Conv2D(16, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(n_classes, activation='softmax')
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

demo = build_cnn_model(X_img.shape[1:], N_CLASSES)
demo.summary()

### 2.2 Five-Fold Stratified Cross-Validation

In [ ]:
kf2 = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
fold_acc_cnn, fold_f1_cnn, history_cnn = [], [], []

print(f'Training CNN (Keras) — {N_SPLITS}-fold stratified CV')
print(f'Architecture: Conv(16)→Pool → Conv(32)→Pool → Dense(64) → Dense({N_CLASSES})\n')
print(f'{"Fold":>6}  {"Val Acc":>10}  {"F1 Macro":>10}  {"Time":>8}')
print('-' * 42)

t_cnn = time.time()
for fold, (tr, te) in enumerate(kf2.split(X_img, y), 1):
    tf.keras.backend.clear_session()
    tf.random.set_seed(RANDOM_STATE + fold)
    fold_start = time.time()

    X_tr_f, y_tr_f = X_img[tr], y[tr]
    X_te_f, y_te_f = X_img[te], y[te]

    model = build_cnn_model(X_img.shape[1:], N_CLASSES)
    hist = model.fit(
        X_tr_f, y_tr_f,
        validation_data=(X_te_f, y_te_f),
        epochs=EPOCHS_CNN, batch_size=BATCH_SIZE,
        callbacks=[callbacks.EarlyStopping(monitor='val_accuracy', patience=4,
                                            restore_best_weights=True, verbose=0)],
        verbose=0
    )
    history_cnn.append(hist)
    y_pred_f = np.argmax(model.predict(X_te_f, verbose=0), axis=1)
    acc = accuracy_score(y_te_f, y_pred_f)
    f1  = f1_score(y_te_f, y_pred_f, average='macro')
    fold_acc_cnn.append(acc); fold_f1_cnn.append(f1)
    print(f'{fold:>6}  {acc*100:>9.2f}%  {f1*100:>9.2f}%  {time.time()-fold_start:>6.1f}s')

cnn_cv_time = time.time() - t_cnn
fold_acc_cnn = np.array(fold_acc_cnn)
fold_f1_cnn  = np.array(fold_f1_cnn)

print(f'\nMean CV Accuracy : {fold_acc_cnn.mean():.4f} ± {fold_acc_cnn.std():.4f}')
print(f'Mean CV F1 Macro : {fold_f1_cnn.mean():.4f} ± {fold_f1_cnn.std():.4f}')
print(f'Total CV Time    : {cnn_cv_time:.1f}s')

### 2.3 Final Model — 80/20 Split for Detailed Report

In [ ]:
tf.keras.backend.clear_session()
X_tr, X_te, y_tr, y_te = train_test_split(
    X_img, y, test_size=0.2,
    stratify=y, random_state=RANDOM_STATE
)

cnn_final = build_cnn_model(X_img.shape[1:], N_CLASSES)
hist_final = cnn_final.fit(
    X_tr, y_tr,
    validation_split=0.1,
    epochs=EPOCHS_CNN, batch_size=BATCH_SIZE,
    callbacks=[callbacks.EarlyStopping(monitor='val_accuracy', patience=5,
                                        restore_best_weights=True, verbose=0)],
    verbose=0
)
y_pred_cnn = np.argmax(cnn_final.predict(X_te, verbose=0), axis=1)

# Save metrics for Exercise 3
cnn_metrics = {
    'name'          : 'CNN',
    'cv_accs'       : fold_acc_cnn,
    'cv_acc_mean'   : fold_acc_cnn.mean(),
    'cv_acc_std'    : fold_acc_cnn.std(),
    'cv_f1_mean'    : fold_f1_cnn.mean(),
    'cv_f1_std'     : fold_f1_cnn.std(),
    'cv_time'       : cnn_cv_time,
    'y_te'          : y_te,
    'y_pred'        : y_pred_cnn,
    'test_acc'      : accuracy_score(y_te, y_pred_cnn),
    'test_f1'       : f1_score(y_te, y_pred_cnn, average='macro'),
    'test_precision': precision_score(y_te, y_pred_cnn, average='macro'),
    'test_recall'   : recall_score(y_te, y_pred_cnn, average='macro'),
}

print('Classification Report (test set, 20%):')
print(classification_report(y_te, y_pred_cnn, target_names=class_names))

### 2.4 Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Accuracy per fold
ax = axes[0]
folds = np.arange(1, N_SPLITS + 1)
ax.bar(folds, fold_acc_cnn*100, color=COLORS[1], alpha=0.85, edgecolor='black')
ax.axhline(fold_acc_cnn.mean()*100, color=COLORS[3], linestyle='--',
           label=f'Mean = {fold_acc_cnn.mean()*100:.1f}%')
ax.set_xticks(folds)
ax.set_xlabel('Fold'); ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 105)
ax.set_title('Accuracy per Fold (5-fold CV)', fontsize=12, fontweight='bold')
ax.legend()

# Learning curves (final model)
ax = axes[1]
h = hist_final.history
ep = range(1, len(h['accuracy']) + 1)
ax.plot(ep, [v*100 for v in h['accuracy']],     color=COLORS[0], lw=2, label='Train')
ax.plot(ep, [v*100 for v in h['val_accuracy']], color=COLORS[2], lw=2,
        ls='--', label='Validation')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
ax.set_title('Learning Curve — Final Model', fontsize=12, fontweight='bold')
ax.legend()

# Confusion matrix
ax = axes[2]
cm_cnn = confusion_matrix(y_te, y_pred_cnn)
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Oranges', ax=ax,
            xticklabels=class_names, yticklabels=class_names)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — CNN', fontsize=12, fontweight='bold')
plt.setp(ax.get_xticklabels(), rotation=40, ha='right')

plt.suptitle('Exercise 2 — CNN for Biome Classification', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('Exercise2.png', dpi=150, bbox_inches='tight')
plt.show()

### 📝 Exercise 2 — Conclusions

The CNN learns directly from raw pixel data and captures spatial patterns — edges, textures, regional structure — that the histogram-based MLP cannot see.

**Observations:**
- The two convolutional blocks act as a hierarchical feature extractor: low-level filters (block 1) detect edges and color regions, while block 2 combines them into textures and shapes characteristic of each biome.
- MaxPooling provides translation invariance — the model recognizes a forest regardless of where the trees appear in the frame.
- Dropout (0.25 after each conv block, 0.4 after the dense layer) is crucial to prevent overfitting given the modest dataset size.
- EarlyStopping typically halts training within 8–12 epochs, indicating the model converges quickly on this task.
- Spatially distinctive biomes (City with rectilinear geometry, Water with smooth texture) tend to be classified with the highest precision.

---
<a id='exercise-3'></a>
## ⚖️ Exercise 3 — Performance Comparison

**Goal:** Compare both models head-to-head using the metrics collected during cross-validation and the final 80/20 evaluation.

We compare:
- **CV Accuracy** and **CV F1 Macro** (mean ± std across 5 folds)
- **Test Accuracy / F1 / Precision / Recall** on the held-out 20% split
- **CV training time** (efficiency)
- **Confusion matrices** side-by-side

### 3.1 Summary Table

In [ ]:
rows = [
    ('CV Accuracy',
     f"{mlp_metrics['cv_acc_mean']*100:.2f} ± {mlp_metrics['cv_acc_std']*100:.2f}%",
     f"{cnn_metrics['cv_acc_mean']*100:.2f} ± {cnn_metrics['cv_acc_std']*100:.2f}%"),
    ('CV F1 Macro',
     f"{mlp_metrics['cv_f1_mean']*100:.2f} ± {mlp_metrics['cv_f1_std']*100:.2f}%",
     f"{cnn_metrics['cv_f1_mean']*100:.2f} ± {cnn_metrics['cv_f1_std']*100:.2f}%"),
    ('Test Accuracy',
     f"{mlp_metrics['test_acc']*100:.2f}%",
     f"{cnn_metrics['test_acc']*100:.2f}%"),
    ('Test F1 Macro',
     f"{mlp_metrics['test_f1']*100:.2f}%",
     f"{cnn_metrics['test_f1']*100:.2f}%"),
    ('Test Precision (Macro)',
     f"{mlp_metrics['test_precision']*100:.2f}%",
     f"{cnn_metrics['test_precision']*100:.2f}%"),
    ('Test Recall (Macro)',
     f"{mlp_metrics['test_recall']*100:.2f}%",
     f"{cnn_metrics['test_recall']*100:.2f}%"),
    ('CV Training Time',
     f"{mlp_metrics['cv_time']:.1f}s",
     f"{cnn_metrics['cv_time']:.1f}s"),
]

print('=' * 65)
print('  COMPARISON: MLP + Histograms  vs  CNN')
print('=' * 65)
print(f'{"Metric":<26} {"MLP":>17} {"CNN":>17}')
print('-' * 65)
for label, v_mlp, v_cnn in rows:
    print(f'  {label:<24} {v_mlp:>17} {v_cnn:>17}')
print('=' * 65)

diff = (cnn_metrics['test_acc'] - mlp_metrics['test_acc']) * 100
winner = 'CNN' if diff > 0 else 'MLP'
print(f'\n🏆 Best on test accuracy: {winner}  (Δ = {abs(diff):.2f}%)')

### 3.2 Visualization

In [ ]:
fig = plt.figure(figsize=(17, 11))
gs  = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.35)

# ── 1. Grouped metric bars ──────────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
labels = ['CV Acc', 'CV F1', 'Test Acc', 'Test F1', 'Precision', 'Recall']
mlp_v = [mlp_metrics[k] for k in ['cv_acc_mean', 'cv_f1_mean', 'test_acc',
                                    'test_f1', 'test_precision', 'test_recall']]
cnn_v = [cnn_metrics[k] for k in ['cv_acc_mean', 'cv_f1_mean', 'test_acc',
                                    'test_f1', 'test_precision', 'test_recall']]
x = np.arange(len(labels)); w = 0.35
b1 = ax1.bar(x - w/2, [v*100 for v in mlp_v], w,
             color=COLORS[0], alpha=0.85, edgecolor='black', label='MLP')
b2 = ax1.bar(x + w/2, [v*100 for v in cnn_v], w,
             color=COLORS[1], alpha=0.85, edgecolor='black', label='CNN')
for bar in list(b1) + list(b2):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8)
ax1.set_xticks(x); ax1.set_xticklabels(labels)
ax1.set_ylabel('Value (%)'); ax1.set_ylim(0, 115)
ax1.set_title('Metrics Comparison: MLP vs CNN', fontsize=12, fontweight='bold')
ax1.legend()

# ── 2. CV accuracy distribution ─────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
data = [mlp_metrics['cv_accs']*100, cnn_metrics['cv_accs']*100]
bp = ax2.boxplot(data, patch_artist=True,
                  medianprops=dict(color=COLORS[3], lw=2))
for patch, c in zip(bp['boxes'], [COLORS[0], COLORS[1]]):
    patch.set_facecolor(c); patch.set_alpha(0.55)
for i, vals in enumerate(data):
    ax2.scatter([i+1]*len(vals), vals,
                color=[COLORS[0], COLORS[1]][i], zorder=5, s=45, alpha=0.9)
ax2.set_xticklabels(['MLP', 'CNN'])
ax2.set_ylabel('CV Accuracy (%)')
ax2.set_title(f'CV Distribution ({N_SPLITS}-fold)', fontsize=12, fontweight='bold')

# ── 3. Confusion matrix MLP ─────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
sns.heatmap(confusion_matrix(mlp_metrics['y_te'], mlp_metrics['y_pred']),
            annot=True, fmt='d', cmap='Blues', ax=ax3,
            xticklabels=class_names, yticklabels=class_names, cbar=False)
ax3.set_xlabel('Predicted'); ax3.set_ylabel('Actual')
ax3.set_title('Confusion — MLP', fontsize=11, fontweight='bold')
plt.setp(ax3.get_xticklabels(), rotation=40, ha='right', fontsize=8)
plt.setp(ax3.get_yticklabels(), fontsize=8)

# ── 4. Confusion matrix CNN ─────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
sns.heatmap(confusion_matrix(cnn_metrics['y_te'], cnn_metrics['y_pred']),
            annot=True, fmt='d', cmap='Oranges', ax=ax4,
            xticklabels=class_names, yticklabels=class_names, cbar=False)
ax4.set_xlabel('Predicted'); ax4.set_ylabel('Actual')
ax4.set_title('Confusion — CNN', fontsize=11, fontweight='bold')
plt.setp(ax4.get_xticklabels(), rotation=40, ha='right', fontsize=8)
plt.setp(ax4.get_yticklabels(), fontsize=8)

# ── 5. Radar chart ──────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2], polar=True)
radar_lbls = ['Acc\n(CV)', 'F1\n(CV)', 'Test\nAcc', 'Test\nF1', 'Prec', 'Recall']
N = len(radar_lbls)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]
mlp_r = mlp_v + mlp_v[:1]; cnn_r = cnn_v + cnn_v[:1]
ax5.plot(angles, mlp_r, color=COLORS[0], lw=2, label='MLP')
ax5.fill(angles, mlp_r, color=COLORS[0], alpha=0.20)
ax5.plot(angles, cnn_r, color=COLORS[1], lw=2, label='CNN')
ax5.fill(angles, cnn_r, color=COLORS[1], alpha=0.20)
ax5.set_xticks(angles[:-1]); ax5.set_xticklabels(radar_lbls, fontsize=8)
ax5.set_ylim(0, 1); ax5.set_yticks([0.25, 0.5, 0.75, 1.0])
ax5.set_yticklabels(['25%', '50%', '75%', '100%'], fontsize=7)
ax5.set_title('Radar Comparison', fontsize=11, fontweight='bold', pad=15)
ax5.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9)

plt.suptitle('Exercise 3 — MLP vs CNN | Biomes of Mexico',
             fontsize=13, fontweight='bold', y=1.00)
plt.savefig('Exercise3.png', dpi=150, bbox_inches='tight')
plt.show()

### 📝 Exercise 3 — Conclusions

**Why does the CNN typically outperform the MLP on this task?**

| Aspect | MLP + Histograms | CNN |
|--------|------------------|-----|
| Feature engineering | Manual (histograms) | Learned end-to-end |
| Information used | **Global** color distribution | Local color + **spatial** structure |
| Translation invariance | N/A (no spatial info) | Yes, via MaxPooling |
| Training speed | Very fast (~seconds) | Slower (~minutes) |
| Parameters | ~50K | ~50K (in this Lite version) |
| Discriminates biomes by | Color palette | Color + texture + shape |

**Where the MLP fails and the CNN succeeds:**
- *Forest vs. Crop* — both green-dominant; their histograms overlap, but their **textures** differ dramatically (irregular canopy vs. parallel rows). Only the CNN sees this.
- *City vs. Mountain* — both gray/brown; CNN distinguishes them via rectilinear geometry vs. organic relief.

**Where the MLP can compete:**
- *Water* and *Desert* have very distinctive color signatures, so the histogram already separates them cleanly.

**Trade-offs:**
- The MLP is a strong, fast, interpretable baseline. If the application is constrained to CPU-only inference, it remains a viable choice.
- The CNN delivers higher accuracy at the cost of training time and computational requirements (GPU strongly recommended for full-scale experiments).
- With a larger dataset and a deeper CNN (32×32 → 64×64 input, 3 conv blocks, batch normalization), the gap would widen further in favor of the CNN.

---
<a id='conclusions'></a>
## 📊 Overall Conclusions

This notebook implemented and evaluated two image classification approaches for satellite images of Mexican biomes. Key takeaways:

| # | Topic | Key Finding |
|---|-------|-------------|
| 1 | MLP + Color Histograms | Strong baseline using only global color distribution; very fast to train; limited by lack of spatial information |
| 2 | Convolutional Neural Network | Learns spatial features directly from pixels; outperforms the MLP especially on biomes with similar color palettes |
| 3 | Head-to-head comparison | The CNN achieves higher accuracy and F1; both models excel on color-distinctive biomes (Water, Desert); CNN's advantage shows on texture-discriminated pairs (Forest vs. Crop) |

**Methodological notes:**
- **Cross-validation is essential**: it gives a more reliable performance estimate than a single train/test split and reveals model stability across data subsets.
- **StandardScaler + early stopping** were critical to keep the MLP from overfitting on the 96-D histogram features.
- **Dropout** was the most impactful regularizer for the CNN on this modestly-sized dataset.
- The Lite CNN configuration (48×48 input, 2 conv blocks, 15 epochs) was chosen for CPU feasibility. With GPU access, a deeper architecture at higher resolution would likely close the remaining gap with state-of-the-art results.

**Final ranking** (by test accuracy): see the comparison table in Exercise 3 — the CNN consistently outperforms the histogram-based MLP, confirming that **spatial information matters** for satellite image classification.

---
*Implemented with Python 3.11 · NumPy · OpenCV · TensorFlow/Keras · scikit-learn · Matplotlib · Seaborn*